In [4]:
from fall_detection_model import Model
import torch
import cv2
import mediapipe

In [5]:
model = Model()
model.load_state_dict(torch.load("ctr-gcn-fall.pth", weights_only=True, map_location="cpu"))
model.eval()

Model(
  (data_bn): BatchNorm1d(99, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (l1): TCN_GCN_unit(
    (gcn1): unit_gcn(
      (convs): ModuleList(
        (0-2): 3 x CTRGC(
          (conv1): Conv2d(3, 8, kernel_size=(1, 1), stride=(1, 1))
          (conv2): Conv2d(3, 8, kernel_size=(1, 1), stride=(1, 1))
          (conv3): Conv2d(3, 64, kernel_size=(1, 1), stride=(1, 1))
          (conv4): Conv2d(8, 64, kernel_size=(1, 1), stride=(1, 1))
          (tanh): Tanh()
        )
      )
      (down): Sequential(
        (0): Conv2d(3, 64, kernel_size=(1, 1), stride=(1, 1))
        (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (soft): Softmax(dim=-2)
      (relu): ReLU(inplace=True)
    )
    (tcn1): MultiScale_TemporalConv(
      (branches): ModuleList(
        (0): Sequential(
          (0): Conv2d(64, 16, kernel_size=(1, 1)

In [6]:
mp_pose  = mediapipe.solutions.pose

pose = mp_pose.Pose(
    model_complexity=1,        # 0 = fast, 1 = balanced, 2 = accurate (chậm hơn)
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

In [22]:
### 1. Đọc video và lấy list_skeleton
video_path = "video4.mp4"
cap = cv2.VideoCapture(video_path)
if not cap.isOpened():
    print("Cannot open camera stream")
else:
    print("Cam opened")

Cam opened


In [23]:
# TODO: lưu giá trị cố định của camera
FPS = cap.get(cv2.CAP_PROP_FPS)
print("FPS: ", FPS)
MAX_FRAME = FPS*10 #

FPS:  25.05921002368401


In [24]:
list_frames = []
skeleton_frames = []
frame_count = 0

while True:
    error_frame_count = 0
    ret, frame = cap.read()
    if not ret:
        error_frame_count += 1
        # TODO: if there are more than 10 continuous error frames, reset variables and break
        if error_frame_count > 10:
            error_frame_count = 0
            frame_count = 0
            skeleton_frames.clear()
            list_frames.clear()
            print("Cannot read the stream")
            break
        continue # dòng này rất quan trọng
    error_frame_count = 0

    rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    result = pose.process(rgb)
    
    if not result.pose_landmarks:
        error_frame_count += 1
        # TODO: if there are more than 10 continuous error frames, reset variables and break
        if error_frame_count > 10:
            error_frame_count = 0
            frame_count = 0
            skeleton_frames.clear()
            list_frames.clear()
            print("break because there is no person")
            break
        continue

    error_frame_count = 0

    joints = []
    for lm in result.pose_landmarks.landmark:
        joints.append([lm.x, lm.y, lm.z])  

    frame_count += 1
    skeleton_frames.append(joints)
    
    if frame_count >= 300 or frame_count >= FPS*10:
        print("Enough")
        break

Enough


In [27]:
len(skeleton_frames)

300

In [28]:
import numpy as np
def process_input(list_skeleton):
    if len(list_skeleton) < 300:
        k = len(list_skeleton)
        joints = []
        for i in range(0, 33):
            joints.append([0.0, 0.0, 0.0])
        while k < 300:
            list_skeleton.append(joints)
            k += 1
    # arr = np.stack([np.array(j).reshape(33, 3) for j in list_skeleton])
    arr = np.array(list_skeleton).astype(np.float32)
    input = torch.from_numpy(arr)
    input = input.permute(2, 0, 1)
    input = input.unsqueeze(0).unsqueeze(-1)

    return input

In [29]:
input = process_input(skeleton_frames)
input.shape

torch.Size([1, 3, 300, 33, 1])

In [ ]:
with torch.inference_mode():
    output = model(input)
    print("output: ", oudtput)
    result = output.argmax(dim=1)
    print("result: ", result)

output:  tensor([[ 1.3442, -2.7063]])
result:  tensor([0])


In [33]:
import os

def count_files(folder_path):
    count = 0
    for root, dirs, files in os.walk(folder_path):
        count += len(files)
    return count


In [37]:
print("Tổng số file:", count_files("E:\\FallVisionDataset-modify\\done\\fall"))
print("Tổng số file:", count_files("E:\\FallVisionDataset-modify\\done\\non-fall"))

Tổng số file: 1961
Tổng số file: 2292


In [42]:
print("Tổng số file:", count_files("E:\\FallVisionDataset-modify-v2\\done\\fall"))
print("Tổng số file:", count_files("E:\\FallVisionDataset-modify-v2\\done\\non-fall"))

Tổng số file: 1957
Tổng số file: 2074
